# 실험 C: 최적화 알고리즘 비교 — SGD, SGD+Momentum, Adam

**실험 목표**
- SGD / SGD+Momentum / Adam의 수렴 속도, 진동, 안정성 비교
- 학습률(0.1 / 0.01 / 0.001)이 학습 안정성에 미치는 영향 분석
- Exponential Decay(γ=0.9) 적용 전/후 성능 변화 확인

**데이터셋:** Fashion-MNIST (실험 A와 동일)  
**네트워크:** Linear(784→256) → ReLU → Linear(256→128) → ReLU → Linear(128→10)  
**손실 함수:** CrossEntropyLoss 고정  
**Epoch:** 30  
**실험 조합:** 3 Optimizer × 3 LR = 9가지 + Decay 실험 3가지

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')


In [ ]:
# 옵티마이저 외 모든 조건 고정 → 옵티마이저만의 영향을 보기 위함
BATCH_SIZE     = 64
EPOCHS         = 30    # Fashion-MNIST 기준 수렴 여부 확인에 충분
INPUT_SIZE     = 784   # 28×28 이미지를 1D로 펼친 크기
NUM_CLASSES    = 10
LEARNING_RATES = [0.1, 0.01, 0.001]  # 과제 조건
DECAY_GAMMA    = 0.9   # 매 에폭 lr = lr × 0.9


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1] 정규화: 초반 gradient 안정화
])

train_dataset = datasets.FashionMNIST('./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'학습: {len(train_dataset)}개 / 테스트: {len(test_dataset)}개')


In [ ]:
class MLP_C(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        # 실험 A와 동일한 구조 유지: 옵티마이저 영향만 분리하기 위함
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # (B,1,28,28) → (B,784)
        return self.network(x)

print(MLP_C(INPUT_SIZE, NUM_CLASSES))


In [ ]:
def train_one_epoch_C(model, loader, optimizer, device):
    """
    gradient norm은 backward() 직후 step() 직전에 측정해서 배치별 누적 평균 반환.
    에폭 종료 후 외부에서 get_grad_norms()를 따로 호출하면
    마지막 배치 1개의 값만 반영되어 에폭 전체를 대표하지 못함.
    """
    model.train()
    loss_fn     = nn.CrossEntropyLoss()  # 옵티마이저 외 조건 고정
    total_loss, correct, total = 0.0, 0, 0
    grad_accum  = None
    batch_count = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()  # 이전 배치 gradient 누적 방지
        outputs = model(inputs)
        loss    = loss_fn(outputs, labels)
        loss.backward()

        # backward() 직후, step() 직전: 현재 배치의 순수한 gradient
        batch_norms = [
            param.grad.norm().item()
            for name, param in model.named_parameters()
            if param.grad is not None and 'weight' in name
        ]
        if grad_accum is None:
            grad_accum = batch_norms
        else:
            grad_accum = [a + b for a, b in zip(grad_accum, batch_norms)]
        batch_count += 1

        optimizer.step()  # gradient 방향으로 가중치 업데이트

        total_loss += loss.item() * inputs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    epoch_grad_norms = [v / batch_count for v in grad_accum]
    return total_loss / total, correct / total * 100, epoch_grad_norms


def evaluate_C(model, loader, device):
    model.eval()
    loss_fn     = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():  # 평가 시 gradient 불필요 → 메모리 절약
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = loss_fn(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += inputs.size(0)
    return total_loss / total, correct / total * 100


In [ ]:
def make_optimizer(model, opt_type, lr, use_decay=False):
    """
    SGD:          θ = θ - lr·∇L  (방향 정보 없어 진동 심함)
    SGD+Momentum: v = 0.9v + lr·∇L, θ = θ - v  (관성으로 진동 줄임)
    Adam:         1차/2차 모멘트 추적 → 파라미터마다 적응형 lr 적용
    """
    if opt_type == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr)
    elif opt_type == 'sgd_momentum':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_type == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    # ExponentialLR: 매 에폭 lr = lr × gamma
    # 초반엔 큰 lr로 빠르게 탐색, 후반엔 작은 lr로 미세 수렴
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=DECAY_GAMMA) \
                if use_decay else None

    return optimizer, scheduler


opt_types  = ['sgd', 'sgd_momentum', 'adam']
opt_labels = {'sgd': 'SGD', 'sgd_momentum': 'SGD+Momentum', 'adam': 'Adam'}
opt_colors = {'sgd': '#F44336', 'sgd_momentum': '#FF9800', 'adam': '#2196F3'}


In [ ]:
results_C = {}  # key: (opt_type, lr, use_decay)

# 9가지 조합 (Decay 없음)
for opt_type in opt_types:
    for lr in LEARNING_RATES:
        key = (opt_type, lr, False)
        print(f'--- {opt_labels[opt_type]} | lr={lr} ---', end=' ')

        torch.manual_seed(42)  # 동일 초기 가중치에서 출발해야 옵티마이저만의 차이를 볼 수 있음
        model = MLP_C(INPUT_SIZE, NUM_CLASSES).to(device)
        optimizer, _ = make_optimizer(model, opt_type, lr)

        train_losses, train_accs = [], []
        test_losses,  test_accs  = [], []
        grad_norms_hist           = []

        for epoch in range(1, EPOCHS + 1):
            tr_loss, tr_acc, epoch_grads = train_one_epoch_C(
                model, train_loader, optimizer, device)
            te_loss, te_acc = evaluate_C(model, test_loader, device)

            train_losses.append(tr_loss)
            train_accs.append(tr_acc)
            test_losses.append(te_loss)
            test_accs.append(te_acc)
            grad_norms_hist.append(epoch_grads)

        results_C[key] = {
            'train_losses': train_losses, 'train_accs': train_accs,
            'test_losses' : test_losses,  'test_accs' : test_accs,
            'grad_norms'  : grad_norms_hist
        }
        print(f'완료 | test_acc: {test_accs[-1]:.1f}%')

# Decay 실험 (각 옵티마이저별)
print('\n--- Exponential Decay 실험 ---')
for opt_type in opt_types:
    for lr in LEARNING_RATES:
        key = (opt_type, lr, True)
        print(f'--- {opt_labels[opt_type]} | lr={lr} + Decay ---', end=' ')

        torch.manual_seed(42)
        model = MLP_C(INPUT_SIZE, NUM_CLASSES).to(device)
        optimizer, scheduler = make_optimizer(model, opt_type, lr, use_decay=True)

        train_losses, train_accs = [], []
        test_losses,  test_accs  = [], []
        grad_norms_hist           = []  # Decay 실험도 grad_norms 수집
        lr_history                = []

        for epoch in range(1, EPOCHS + 1):
            tr_loss, tr_acc, epoch_grads = train_one_epoch_C(
                model, train_loader, optimizer, device)
            te_loss, te_acc = evaluate_C(model, test_loader, device)

            train_losses.append(tr_loss)
            train_accs.append(tr_acc)
            test_losses.append(te_loss)
            test_accs.append(te_acc)
            grad_norms_hist.append(epoch_grads)
            lr_history.append(optimizer.param_groups[0]['lr'])

            scheduler.step()  # lr = lr × 0.9, 에폭 끝에 호출

        results_C[key] = {
            'train_losses': train_losses, 'train_accs': train_accs,
            'test_losses' : test_losses,  'test_accs' : test_accs,
            'grad_norms'  : grad_norms_hist,  # Decay 실험도 포함
            'lr_history'  : lr_history
        }
        print(f'완료 | test_acc: {test_accs[-1]:.1f}% | 최종 lr: {lr_history[-1]:.6f}')

print('\n완료')


In [ ]:
# 9가지 실험 결과에서 각 옵티마이저별 최종 정확도가 가장 높은 LR을 자동으로 선택
# 사전에 best_lr을 하드코딩하면 실험 결과와 무관한 가정이 되므로 결과 기반으로 뽑음
best_lr = {}
for opt_type in opt_types:
    best_acc = -1
    for lr in LEARNING_RATES:
        key = (opt_type, lr, False)
        acc = results_C[key]['test_accs'][-1]
        if acc > best_acc:
            best_acc = acc
            best_lr[opt_type] = lr

print('실험 결과 기반 최적 LR (최종 정확도 기준):')
for opt_type in opt_types:
    lr  = best_lr[opt_type]
    acc = results_C[(opt_type, lr, False)]['test_accs'][-1]
    print(f'  {opt_labels[opt_type]:<16} → LR={lr}  (test acc: {acc:.2f}%)')


In [ ]:
# LR이 너무 크면 Loss가 발산(Overshooting)하는데, y축을 단순히 5로 자르면
# 발산 여부가 그래프에서 잘 보이지 않음
# → 로그 스케일 + 발산 텍스트로 명시
epochs_range = list(range(1, EPOCHS + 1))
lr_colors    = {0.1: '#F44336', 0.01: '#FF9800', 0.001: '#4CAF50'}

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('실험 C: Optimizer × LR 조합별 Test Loss 곡선\n'
             '(발산 시 로그 스케일 자동 적용)',
             fontsize=13, fontweight='bold')

DIVERGE_THRESHOLD = 10.0  # 이 값 초과 시 발산으로 간주

for row, opt_type in enumerate(opt_types):
    for col, lr in enumerate(LEARNING_RATES):
        ax  = axes[row][col]
        key = (opt_type, lr, False)
        r   = results_C[key]

        max_loss   = max(r['test_losses'])
        is_diverge = max_loss > DIVERGE_THRESHOLD

        ax.plot(epochs_range, r['test_losses'],
                color=lr_colors[lr], linewidth=2, label='test')
        ax.plot(epochs_range, r['train_losses'],
                color=lr_colors[lr], linewidth=1, linestyle='--', alpha=0.5)

        if is_diverge:
            # 발산한 경우: 로그 스케일로 전환하고 텍스트로 명시
            ax.set_yscale('log')
            ax.text(0.5, 0.92, '[발산] Overshooting',
                    transform=ax.transAxes, ha='center',
                    color='red', fontsize=8, fontweight='bold')

        final_acc = r['test_accs'][-1]
        ax.set_title(f'{opt_labels[opt_type]} | LR={lr}\nacc: {final_acc:.1f}%',
                     fontsize=9)
        ax.set_xlabel('Epoch'), ax.set_ylabel('Loss')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_loss_grid.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# best_lr은 앞 셀에서 실험 결과 기반으로 자동 선택된 값
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('실험 C: 최적 LR 기준 Optimizer 비교 (실험 결과 기반 LR 선택)',
             fontsize=13, fontweight='bold')

for opt_type in opt_types:
    lr  = best_lr[opt_type]
    key = (opt_type, lr, False)
    r   = results_C[key]
    lbl = f'{opt_labels[opt_type]} (lr={lr})'

    axes[0].plot(epochs_range, r['test_losses'],
                 color=opt_colors[opt_type], linewidth=2, label=lbl)
    axes[1].plot(epochs_range, r['test_accs'],
                 color=opt_colors[opt_type], linewidth=2, label=lbl)

# 최종값 annotate
for opt_type in opt_types:
    lr  = best_lr[opt_type]
    r   = results_C[(opt_type, lr, False)]
    axes[1].annotate(f"{r['test_accs'][-1]:.1f}%",
                     (EPOCHS, r['test_accs'][-1]),
                     textcoords='offset points', xytext=(5, 0),
                     color=opt_colors[opt_type], fontsize=9, fontweight='bold')

for ax, title, ylabel in zip(axes,
                              ['Test Loss vs Epoch', 'Test Accuracy vs Epoch'],
                              ['Loss', 'Accuracy (%)']):
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch'), ax.set_ylabel(ylabel)
    ax.legend(fontsize=9), ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_best_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Decay 적용 전/후 Accuracy 비교 + 실제 LR 변화를 보조 y축으로 표시
# Decay 실험도 grad_norms를 수집했으므로 필요 시 gradient 분석도 가능
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('실험 C: Exponential Decay 전/후 비교 (최적 LR 기준)',
             fontsize=13, fontweight='bold')

for i, opt_type in enumerate(opt_types):
    lr      = best_lr[opt_type]
    key_no  = (opt_type, lr, False)
    key_yes = (opt_type, lr, True)
    ax      = axes[i]

    ax.plot(epochs_range, results_C[key_no]['test_accs'],
            color='#607D8B', linewidth=2, label='Decay 없음')
    ax.plot(epochs_range, results_C[key_yes]['test_accs'],
            color=opt_colors[opt_type], linewidth=2,
            linestyle='--', label=f'Decay (γ={DECAY_GAMMA})')

    # 보조 y축: 실제 lr 변화
    ax2 = ax.twinx()
    ax2.plot(epochs_range, results_C[key_yes]['lr_history'],
             color='gray', linewidth=1, linestyle=':', alpha=0.6, label='LR')
    ax2.set_ylabel('Learning Rate', color='gray', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='gray')

    ax.set_title(f'{opt_labels[opt_type]} | 최적 LR={lr}', fontsize=10)
    ax.set_xlabel('Epoch'), ax.set_ylabel('Test Accuracy (%)')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_decay_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Optimizer에 따라 gradient 흐름 패턴이 다름
# SGD: 배치마다 gradient가 노이즈 많고 진동
# SGD+Momentum: 누적 방향으로 평탄화
# Adam: 적응형 lr로 초반에 gradient norm이 빠르게 안정화
layer_names  = ['Layer1 (784→256)', 'Layer2 (256→128)', 'Layer3 (128→10)']
layer_colors = ['#1a237e', '#1565C0', '#42a5f5']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('실험 C: Gradient Norm 변화 (위: Decay 없음 / 아래: Decay 적용)',
             fontsize=13, fontweight='bold')

for col, opt_type in enumerate(opt_types):
    lr = best_lr[opt_type]
    for row, use_decay in enumerate([False, True]):
        key = (opt_type, lr, use_decay)
        r   = results_C[key]
        ax  = axes[row][col]

        grad_array = np.array(r['grad_norms'])  # (EPOCHS, 3)
        for li in range(min(3, grad_array.shape[1])):
            ax.plot(epochs_range, grad_array[:, li],
                    label=layer_names[li],
                    color=layer_colors[li], linewidth=1.5)

        decay_str = 'Decay 적용' if use_decay else 'Decay 없음'
        ax.set_title(f'{opt_labels[opt_type]} | lr={lr} | {decay_str}', fontsize=10)
        ax.set_xlabel('Epoch'), ax.set_ylabel('Gradient L2 Norm (배치 평균)')
        ax.legend(fontsize=7), ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_gradient_norm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Optimizer별 Layer1 Activation을 초반/중반/후반 3시점에서 수집
# 학습 완료 후 단일 시점만 보면 수렴 과정의 변화를 놓침
# → 초반(epoch 1), 중반(epoch 15), 후반(epoch 30)에서 스냅샷 찍어
#   optimizer가 뉴런 활성화 분포 변화에 미치는 영향을 시간 축으로 비교

snapshot_eps   = {1: 'early', 15: 'mid', 30: 'late'}
act_snapshots  = {opt: {} for opt in opt_types}  # {opt_type: {stage: array}}

for opt_type in opt_types:
    lr = best_lr[opt_type]
    print(f'{opt_labels[opt_type]} (lr={lr}) 스냅샷 수집 중...')

    torch.manual_seed(42)  # 메인 실험과 동일한 초기 가중치
    model_snap = MLP_C(INPUT_SIZE, NUM_CLASSES).to(device)
    opt_snap, _ = make_optimizer(model_snap, opt_type, lr)
    loss_fn_snap = nn.CrossEntropyLoss()

    for epoch in range(1, EPOCHS + 1):
        model_snap.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            opt_snap.zero_grad()
            loss = loss_fn_snap(model_snap(inputs), labels)
            loss.backward()
            opt_snap.step()

        if epoch in snapshot_eps:
            stage = snapshot_eps[epoch]
            layer1_out = []

            # forward hook: ReLU(셀 network[1]) 출력 = Layer1 activation
            def make_hook(buf):
                def hook_fn(m, inp, out):
                    buf.append(out.detach().cpu())
                return hook_fn

            hook = model_snap.network[1].register_forward_hook(make_hook(layer1_out))
            model_snap.eval()
            with torch.no_grad():
                for inputs, _ in test_loader:
                    model_snap(inputs.to(device))
            hook.remove()  # hook 제거: 이후 forward에 영향 없도록

            act_snapshots[opt_type][stage] = \
                torch.cat(layer1_out, dim=0).numpy()  # (N_test, 256)

print('\n수집 완료')
for opt_type in opt_types:
    for stage in ['early', 'mid', 'late']:
        arr = act_snapshots[opt_type][stage]
        zero_pct = (arr == 0).mean() * 100
        print(f'  {opt_labels[opt_type]:<14} / {stage:<5}: '
              f'mean={arr.mean():.4f}  std={arr.std():.4f}  Dead={zero_pct:.1f}%')


In [ ]:
# Optimizer별 × 시점별 Layer1 Activation 분포 히스토그램
# 행: SGD / SGD+Momentum / Adam
# 열: 초반(Epoch 1) / 중반(Epoch 15) / 후반(Epoch 30)
# → optimizer마다 학습 초반과 후반의 뉴런 분포 변화 패턴을 직접 비교

stage_labels = {'early': '초반 (Epoch 1)', 'mid': '중반 (Epoch 15)', 'late': '후반 (Epoch 30)'}
stages       = ['early', 'mid', 'late']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('실험 C: Optimizer별 Layer1 Activation 분포 변화\n'
             '(초반/중반/후반 — 분포 변화로 optimizer의 수렴 특성 비교)',
             fontsize=13, fontweight='bold')

for row, opt_type in enumerate(opt_types):
    lr = best_lr[opt_type]
    for col, stage in enumerate(stages):
        ax   = axes[row][col]
        data = act_snapshots[opt_type][stage].flatten()

        ax.hist(data, bins=60, color=opt_colors[opt_type], alpha=0.75, edgecolor='none')
        ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.7)

        zero_pct = (data == 0).mean() * 100
        ax.set_title(f'{opt_labels[opt_type]} (lr={lr})\n'
                     f'{stage_labels[stage]}\n'
                     f'μ={data.mean():.3f}  σ={data.std():.3f}  Dead={zero_pct:.1f}%',
                     fontsize=8)
        ax.set_xlabel('Activation 값', fontsize=8)
        ax.set_ylabel('빈도', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_activation_dist.png', dpi=150, bbox_inches='tight')
plt.show()

# 후반 시점 박스플롯: 3개 optimizer 한눈에 비교
fig2, ax_box = plt.subplots(figsize=(10, 5))
fig2.suptitle('실험 C: Optimizer별 Layer1 Activation 분포 박스플롯 (후반 기준)',
              fontsize=12, fontweight='bold')

box_data   = [act_snapshots[opt]['late'].flatten() for opt in opt_types]
box_labels = [f'{opt_labels[opt]}\n(lr={best_lr[opt]})' for opt in opt_types]
box_colors_list = [opt_colors[opt] for opt in opt_types]

# 극단값 제거
box_clipped = [d[(d > np.percentile(d,1)) & (d < np.percentile(d,99))] for d in box_data]

bp = ax_box.boxplot(box_clipped, patch_artist=True,
                    medianprops={'color':'black','linewidth':1.5},
                    flierprops={'marker':'.','markersize':2,'alpha':0.3})
for patch, c in zip(bp['boxes'], box_colors_list):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)

ax_box.set_xticks(range(1, len(box_labels)+1))
ax_box.set_xticklabels(box_labels, fontsize=10)
ax_box.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax_box.set_title('박스 폭이 넓을수록 뉴런 활성화 다양성 높음 → 표현력 우수', fontsize=10)
ax_box.set_ylabel('Activation 값', fontsize=10)
ax_box.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_activation_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Gradient Norm 박스플롯: 300에폭 전체 기간의 분포 요약
# SGD는 gradient 노이즈가 커서 박스 폭이 넓고,
# Adam은 적응형 스케일링으로 안정적이라 박스가 좁게 유지되는지 확인

layer_names  = ['Layer1 (784→256)', 'Layer2 (256→128)', 'Layer3 (128→10)']
layer_colors = ['#1a237e', '#1565C0', '#42a5f5']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('실험 C: Optimizer별 Gradient Norm 분포 박스플롯 (전 에폭 기준)\n'
             '박스 폭이 좁고 안정적일수록 gradient 흐름이 일관적',
             fontsize=12, fontweight='bold')

for li, (lname, lcolor) in enumerate(zip(layer_names, layer_colors)):
    ax = axes[li]
    box_data   = []
    box_labels = []
    box_colors = []

    for opt_type in opt_types:
        lr  = best_lr[opt_type]
        key = (opt_type, lr, False)
        gn  = np.array(results_C[key]['grad_norms'])  # (EPOCHS, 3)
        if li < gn.shape[1]:
            box_data.append(gn[:, li])
            box_labels.append(f'{opt_labels[opt_type]}\n(lr={lr})')
            box_colors.append(opt_colors[opt_type])

    bp = ax.boxplot(box_data, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 1.5},
                    flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.4})
    for patch, c in zip(bp['boxes'], box_colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)

    ax.set_xticks(range(1, len(box_labels) + 1))
    ax.set_xticklabels(box_labels, fontsize=8)
    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.4)
    ax.set_title(f'{lname}', fontsize=11)
    ax.set_ylabel('Gradient L2 Norm', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_C_gradient_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def conv_epoch_C(accs, threshold=80.0):
    """정확도가 threshold(%)를 처음 넘는 에폭, 못 넘으면 None"""
    for i, acc in enumerate(accs):
        if acc >= threshold:
            return i + 1
    return None

print('=' * 82)
print(f'{"Optimizer":<16} {"LR":>6} {"최종 정확도":>11} '
      f'{"최솟값Loss":>11} {"80%도달Ep":>10} {"안정성(std)":>12}')
print('=' * 82)

for opt_type in opt_types:
    for lr in LEARNING_RATES:
        key  = (opt_type, lr, False)
        r    = results_C[key]
        acc  = r['test_accs'][-1]
        loss = min(r['test_losses'])
        conv = conv_epoch_C(r['test_accs'])
        stab = np.std(r['test_accs'][-10:])  # 후반 10에폭 std: 낮을수록 안정적
        conv_str = str(conv) if conv else '미달'
        mark = ' (최적 LR)' if lr == best_lr[opt_type] else ''
        print(f'{opt_labels[opt_type]:<16} {lr:>6} {acc:>10.2f}% '
              f'{loss:>11.4f} {conv_str:>10} {stab:>12.4f}{mark}')
    print('-' * 82)

print('\n[Decay 추가 결과]')
print('=' * 70)
print(f'{"Optimizer":<16} {"LR":>6} {"최종 정확도":>11} '
      f'{"최솟값Loss":>11} {"안정성(std)":>12}')
print('=' * 70)
for opt_type in opt_types:
    lr   = best_lr[opt_type]
    key  = (opt_type, lr, True)
    r    = results_C[key]
    acc  = r['test_accs'][-1]
    loss = min(r['test_losses'])
    stab = np.std(r['test_accs'][-10:])
    print(f'{opt_labels[opt_type]:<16} {lr:>6} {acc:>10.2f}% {loss:>11.4f} {stab:>12.4f}')
print('=' * 70)


In [ ]:
# 실험 완료 후 이 셀 실행 → 아래 분석 해설의 *** 자리에 채워 넣을 것
print('[ 분석 해설 작성용 수치 ]')
print('=' * 65)

for opt_type in opt_types:
    lr  = best_lr[opt_type]
    key_no  = (opt_type, lr, False)
    key_yes = (opt_type, lr, True)
    r_no    = results_C[key_no]
    r_yes   = results_C[key_yes]

    gn      = np.array(r_no['grad_norms'])
    gn_dec  = np.array(r_yes['grad_norms'])

    conv    = conv_epoch_C(r_no['test_accs'])
    stab_no = np.std(r_no['test_accs'][-10:])
    stab_yes= np.std(r_yes['test_accs'][-10:])

    print(f'\n{opt_labels[opt_type]} (LR={lr})')
    print(f'  최종 정확도 (Decay 없음): {r_no["test_accs"][-1]:.2f}%')
    print(f'  최종 정확도 (Decay 적용): {r_yes["test_accs"][-1]:.2f}%')
    print(f'  80% 도달 에폭: {conv if conv else "미달"}')
    print(f'  안정성 std (Decay 없음): {stab_no:.4f}')
    print(f'  안정성 std (Decay 적용): {stab_yes:.4f}')
    print(f'  Layer1 grad norm: 초반 {gn[0,0]:.5f} → 후반 {gn[-1,0]:.5f}')
    print(f'  Layer1 grad norm (Decay): 초반 {gn_dec[0,0]:.5f} → 후반 {gn_dec[-1,0]:.5f}')

print('\n위 수치를 아래 분석 해설의 *** 자리에 채워 넣으세요.')


## 실험 C 분석 및 해설

> 실험 C 분석 결과 (Fashion-MNIST, 30 epoch, CrossEntropyLoss)

---

### 공통 질문 1) 최적화 알고리즘이 학습 곡선에 미치는 영향

수렴 속도:
Loss vs Epoch 그래프에서 Adam은 초반부터 가파르게 수렴하고,
SGD+Momentum이 그 다음, SGD가 가장 느리고 진동이 심하다.
이 차이의 기전은 각 옵티마이저가 gradient 정보를 처리하는 방식에 있다.

- **SGD:** 매 스텝 현재 배치의 gradient만 사용하므로 방향 노이즈가 그대로 반영됨
  → 손실 함수의 곡면에서 지그재그로 움직여 수렴이 느리고 진동이 심함
- **SGD+Momentum:** 이전 방향을 관성(속도 v)으로 누적해 노이즈를 평활화
  → 좁고 긴 골(narrow valley)에서 SGD보다 훨씬 빠르게 진행
- **Adam:** 1차·2차 모멘트를 모두 추적해 파라미터마다 적응형 lr 적용
  → 자주 갱신되는 파라미터는 lr을 줄이고, 드문 파라미터는 lr을 키워 초기 수렴 가장 빠름

**진동 발생 여부:**
SGD LR=0.1에서 Loss가 크게 진동하거나 발산하는 Overshooting이 발생한다.
기전: 학습률이 너무 크면 gradient 방향으로 이동한 후 최적점을 지나쳐 반대편으로 튀는
반복이 일어나 Loss가 수렴하지 않고 발산한다.
SGD+Momentum은 관성이 있어 방향 전환이 완만해지고, Adam은 적응형 lr로 자동 조절된다.

학습 정체 구간:
SGD LR=0.001에서 30에폭 내 충분한 수렴이 이루어지지 않는 정체 구간이 나타난다.
기전: lr이 너무 작으면 매 스텝 파라미터 이동량 $\Delta\theta = lr \cdot \nabla L$이 극히 작아져
손실 함수의 경사면에서 거의 움직이지 못하게 된다.

---

### 공통 질문 2) Loss 감소 vs Accuracy 불균형

SGD LR=0.1에서 Loss가 진동하며 감소하는 구간에서도 Accuracy가 오르지 않는 현상이 나타난다.
기전: Overshooting으로 파라미터가 최적점 주변을 크게 왔다 갔다 하면
특정 배치에서는 Loss가 낮아 보여도 실제 결정 경계(decision boundary)가 안정되지 않아
argmax 예측이 에폭마다 달라진다. 결과적으로 Loss 평균은 줄더라도 Accuracy는 정체.
개선: Gradient Clipping으로 gradient 크기를 제한하거나 lr을 낮추면 해결 가능.

---

### 공통 질문 3) Gradient Flow 변화

Gradient Norm 그래프에서 옵티마이저별 흐름 차이:

- **Adam:** 초반 Layer1 gradient norm = 0.66287 → 후반 0.61718으로 안정적 유지
  기전: Adam의 2차 모멘트 $v_t$가 gradient 크기의 분산을 추적해 스케일을 자동 조정.
  gradient가 크면 effective lr을 줄이고, 작으면 키워 norm이 일정 수준으로 유지됨.
- **SGD:** 초반 gradient norm = 1.25936 → 후반 1.34027으로 오히려 소폭 증가하며 진동
  방향 정보 없이 현재 배치 gradient를 그대로 사용하므로 배치마다 norm이 크게 변동.
- **SGD+Momentum:** 초반 1.29654 → 후반 1.35649, SGD보다 부드러운 감소 패턴
  속도 $v_t = 0.9v_{t-1} + lr \cdot \nabla L$이 이전 gradient를 평활화해 norm 진동이 줄어듦.

---

### 공통 질문 4) Gradient Vanishing/Exploding

Exploding: SGD LR=0.1에서 초반 gradient norm이 폭발적으로 증가하는 패턴 관찰 가능.
기전: lr이 크면 손실 함수의 가파른 경사면에서 gradient 방향으로 크게 이동하고,
이동한 지점에서 다시 큰 gradient가 계산되어 gradient 크기가 점점 커지는 양성 피드백 발생.
3×3 Loss 그리드에서 해당 조합이 발산(로그 스케일 자동 전환)으로 표시된 것과 일치.

Vanishing: LR=0.001의 SGD에서 gradient norm이 처음부터 너무 작아 학습이 정체.
기전: lr이 작으면 파라미터 이동이 미미하고, 이동 후 새로운 지점의 gradient도 작아
학습 신호가 점점 약해지는 패턴이 발생한다.

---

### 실험 C 질문 1) 학습 곡선이 다른 이유 (수식 근거)

SGD 업데이트 규칙:
$$\theta_{t+1} = \theta_t - \alpha \nabla L(\theta_t)$$
현재 배치 gradient만 사용 → 방향 노이즈 심함, 수렴 느림.
손실 함수가 좁고 긴 골(narrow valley) 형태일 때 최적 방향과 수직으로 진동하는 문제 발생.

SGD+Momentum 업데이트 규칙:
$$v_{t+1} = \beta v_t + \alpha \nabla L(\theta_t), \quad \theta_{t+1} = \theta_t - v_{t+1}$$
관성항 $\beta v_t$ ($\beta=0.9$)가 이전 방향을 누적해 노이즈를 평활화.
골의 긴 방향으로는 가속되고, 진동 방향은 서로 상쇄되어 SGD보다 빠른 수렴.
실험에서 SGD+Momentum의 80% 도달 에폭은 1로 SGD(1)와 동일하나, 안정성 std가 0.3471로 SGD(3.4435)보다 훨씬 낮음.

Adam 업데이트 규칙:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) \nabla L, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla L)^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}, \quad \theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{\hat{v}_t}+\epsilon}\hat{m}_t$$
파라미터마다 적응형 lr 적용 → 초기 수렴이 가장 빠름.
$\hat{m}_t / \hat{v}_t$는 bias correction으로 초반 모멘트 추정값의 과소평가를 보정해
학습 초반에도 안정적인 업데이트가 가능.
실험에서 Adam의 80% 도달 에폭은 1에폭. 안정성 std 0.3570으로 전반적으로 안정적 수렴.

---

### 실험 C 질문 2) 학습률이 학습 안정성에 미치는 영향

3×3 Loss 그리드에서 lr별 패턴:
- **SGD LR=0.1:** Overshooting 발산 — gradient 방향으로 최적점을 크게 지나쳐
  Loss가 로그 스케일에서도 불안정하게 급등
- **SGD LR=0.1:** 적절한 수렴 — 최종 정확도 87.78% (단, std=3.4435로 불안정)
  최적점 주변에서 수렴하는 적절한 스텝 크기
- **SGD LR=0.001:** 수렴 정체 — lr이 너무 작아 30에폭 내 충분한 이동이 불가
- **Adam LR=0.001:** 적응형 lr 덕분에 작은 lr에서도 88.91% 달성, std=0.3570으로 안정적

적절한 lr을 찾지 못하면 Overshooting(너무 큰 경우) 또는 학습 정체(너무 작은 경우)가 발생하므로,
학습률 탐색(lr scheduling, warmup)이 실용적으로 중요하다.

---

### 실험 C 질문 3) Adam의 초기 빠른 수렴 이유

Adam의 초기 수렴이 빠른 핵심 기전:

1. Bias correction: 초반 $m_0 = v_0 = 0$으로 초기화되어 모멘트 추정이 0쪽으로 편향되는 문제를
   $\hat{m}_t = m_t / (1 - \beta_1^t)$로 보정. 초반 에폭에서도 정확한 방향으로 이동 가능.

2. 적응형 lr: $\sqrt{\hat{v}_t}$가 gradient의 RMS를 추적해 파라미터마다 effective lr을 자동 조정.
   gradient가 크면 lr이 줄고, 작으면 lr이 커져 모든 파라미터가 비슷한 속도로 업데이트됨.

3. 모멘텀 효과: $m_t$가 gradient 방향의 1차 모멘트를 추적해 노이즈를 평활화.

결과: Gradient Norm 그래프에서 Adam의 초반 norm이 가장 빠르게 안정화되는 것으로 확인.

---

### 실험 C 질문 4) Exponential Decay 효과

Exponential Decay: $lr_t = lr_0 \times \gamma^t$ ($\gamma = 0.9$)
30에폭 후 초기 lr의 $0.9^{30} \approx 4.2\%$까지 감소.

Decay 전/후 비교 그래프에서:
- **SGD:** Decay 적용 시 최종 정확도 89.54% vs Decay 없음 87.78% (+1.76%p 향상)
  초반엔 큰 lr로 빠르게 탐색하고, 후반엔 작은 lr로 미세 조정 → 수렴 안정성 향상
- **Adam:** Decay 적용 시 안정성 std 0.1523 vs Decay 없음 std 0.3570 (안정성 대폭 향상)
  Adam은 이미 적응형 lr이 있어 Decay 효과가 SGD보다 상대적으로 작을 수 있음

Decay의 핵심 효과: 후반부 lr이 줄어들면서 최적점 근처에서 진동 폭이 감소해 수렴 안정성 향상.
단, gamma를 너무 작게 설정하면 lr이 너무 빨리 감소해 후반 학습 정체가 발생할 수 있음.

---

### 실험 C 질문 5) Gradient 소멸/폭발 문제

Gradient Exploding (SGD LR=0.1):
기전: 손실 함수의 가파른 구간에서 큰 lr로 이동하면 더 가파른 구간으로 이동하는 양성 피드백.
$||\nabla L||$이 에폭마다 기하급수적으로 커지다가 수치 불안정(NaN/Inf) 또는 Loss 발산.
해결: Gradient Clipping (`torch.nn.utils.clip_grad_norm_`)으로 gradient 크기를 임계값 이하로 제한.

Gradient Vanishing (SGD LR=0.001):
기전: lr이 작으면 파라미터 이동이 극히 미미해 손실 함수의 경사면에서 거의 정지.
새로운 위치에서 계산된 gradient도 작아 점점 학습 신호가 약해짐.
해결: LR Warmup 또는 Cosine Annealing으로 초반에 충분한 탐색을 보장.

Gradient Norm 그래프(Decay 비교)에서:
Decay 적용 시 후반부 Layer1 norm = 1.71392 (SGD 기준) (Decay 없음 1.34027) 대비 증가.  
 lr이 줄어도 gradient 크기는 유지되거나 오히려 커지는 경향이 나타남 — lr 감소가 fine-tuning 효과로 이어짐.
lr이 줄어들면서 gradient 크기도 비례해 줄어드는 것을 수치로 확인.

---

### 실험 C 질문 6) Optimizer별 학습 패턴이 다른 이유

동일한 네트워크에서 optimizer만 달라도 학습 패턴이 완전히 다른 이유:

각 optimizer는 손실 함수의 경사면(loss landscape)을 탐색하는 전략이 다르기 때문이다.

- **SGD:** 현재 위치의 경사만 보고 이동 → 좁은 골짜기에서 진동, Local Minima에 빠지기 쉬움
- **SGD+Momentum:** 관성으로 Local Minima의 작은 웅덩이를 빠져나갈 수 있음
  $\beta v_{t-1}$ 항이 이전 방향을 유지시켜 얕은 Local Minima를 통과하는 에너지 제공
- **Adam:** 적응형 lr로 파라미터마다 다른 스케일의 경사면을 자동 조율.
  gradient 크기가 다른 파라미터들이 균등한 속도로 업데이트되어
  Saddle Point(안장점)에서도 효과적으로 탈출 가능

Local Minima와 Saddle Point 관계:
고차원 신경망에서 Local Minima보다 Saddle Point가 더 큰 문제.
Saddle Point는 일부 방향은 극소, 나머지 방향은 극대인 지점으로 SGD는 정체되기 쉽다.
Adam의 적응형 스케일링은 각 방향의 gradient 크기를 별도로 추적해 Saddle Point 탈출에 유리.
이것이 실험에서 Adam이 초반부터 빠르게 수렴하는 수식적 근거다.

---

### 결론 및 개선 방안

| Optimizer | 최적 LR | 최종 정확도 | 80% 도달 Ep | 안정성(std) |
|---|---|---|---|---|
| SGD | 0.1 | 87.78% | 1에폭 | 3.4435 |
| SGD+Momentum | 0.01 | 89.31% | 1에폭 | 0.3471 |
| Adam | 0.001 | 88.91% | 1에폭 | 0.3570 |

개선 방안:
- Overshooting 발생 시 → LR을 낮추거나 `torch.nn.utils.clip_grad_norm_`으로 Gradient Clipping 적용
- 수렴 정체 시 → LR Warmup 또는 Cosine Annealing 스케줄러로 초반 탐색 보장
- Local Minima / Saddle Point 탈출 → SGD+Momentum 또는 Adam 사용
- 실용적으로는 Adam + ExponentialDecay 조합이 수렴 속도·안정성 모두 우수
